# Localisation of solvers

So far, all solvers in our tutorials have been active everywhere: Each
solver holds its data in every (unrefined) cell of the mesh, and it updates
these data in each and every time step. *Localisation* is the opposite: A
solver is active only in a part of the mesh, i.e. it neither stores nor updates
any data elsewhere. Localisation comes in two flavours:

- **Localisation in space.** A solver is active only within a subdomain. The
  classic example is a posteriori limiting, where a robust Finite Volume
  solver takes over from a higher-order solver only where shocks arise.
- **Localisation in resolution.** A solver is active only on a certain mesh
  level or mesh resolution. Peano's spacetree holds all resolution levels
  anyway, i.e. we can tell one solver to live on one particular level, while
  another solver lives on a finer level of the very same mesh.

We start with the motivation for the latter flavour, before we discuss the
mechanics of localisation along the lines of the former. As soon as solvers
are localised, they typically have to exchange data where their active
regions meet or overlap. This is discussed in
@ref tutorials_exahype2_45_coupling.

## Localisation in resolution and uncertainty quantification

Localisation in resolution might look like an exotic feature, but it is a key
ingredient of many modern uncertainty quantification (UQ) methods. In UQ, we do
not run one simulation, but many: We are interested in the statistics of some
output quantity $Q$, such as the expected wave height at a buoy, where the input
data (initial conditions, material parameters, bathymetry, ...) are uncertain.
A plain Monte Carlo estimate averages $Q$ over $N$ samples. Its statistical
error decays with $N^{-1/2}$ only, so we need many samples, and each sample is a
full PDE solve on a fine mesh. This quickly becomes prohibitively expensive.

Multilevel Monte Carlo (MLMC) (Giles 2008; Giles 2015) reduces this cost by
exploiting a hierarchy of meshes $\ell = 0, 1, \ldots, L$ from coarse to fine.
It rewrites the expectation on the finest mesh as a telescoping sum

\begin{equation*}
\mathbb{E}[Q_L] = \mathbb{E}[Q_0] + \sum_{\ell=1}^{L} \mathbb{E}[Q_\ell - Q_{\ell-1}]
\end{equation*}

and estimates each term with its own number of samples. Most samples are taken
on the cheap coarse meshes. The corrections $Q_\ell - Q_{\ell-1}$ have a small
variance and hence need only a few of the expensive fine-mesh samples.
Mishra, Schwab and Šukys (2012) have shown that this pays off for nonlinear
hyperbolic systems of conservation laws such as the Euler or the shallow water
equations, i.e. for exactly the type of problems that ExaHyPE is written for.
Multi-index Monte Carlo (Haji-Ali, Nobile and Tempone 2016) generalises the
idea to several hierarchies at once, such as the mesh size and the time step
size.

The same idea carries over to Bayesian inverse problems, where we infer
uncertain parameters from measurements through Markov chain Monte Carlo (MCMC):
Multilevel MCMC (Dodwell et al. 2015) and multilevel delayed acceptance
(Lykkegaard et al. 2023) use cheap coarse-mesh solves to steer or screen
proposals before the expensive fine-mesh solve is invoked. Seelinger et al.
(2021) have parallelised multilevel MCMC on supercomputers and identified the
source of a tsunami from buoy data, using ExaHyPE (Reinarz et al. 2020) as
forward solver. All of these approaches belong to the wider family of
multi-fidelity methods (Peherstorfer, Willcox and Gunzburger 2018), which
combine cheap low-fidelity models with expensive high-fidelity ones.

Where does localisation come into play? The crucial term in the telescoping
sum is the correction $Q_\ell - Q_{\ell-1}$: Both $Q_\ell$ and $Q_{\ell-1}$ have
to be computed for the very same sample, i.e. with the very same uncertain
input. Otherwise, the two do not correlate, and the variance of their
difference is not small. Typical UQ codes realise this through two independent
simulations, one per mesh, and they have to ensure that both see the same
input. With localisation in resolution, we can realise both simulations within
one mesh instead: One solver instance lives on level $\ell$ of the spacetree,
the other one on level $\ell-1$. Two adjacent levels of Peano's spacetree
differ by a factor of three in the mesh size, which yields a natural level
hierarchy. Both solvers share the mesh, the input data, the parallelisation
and the domain decomposition, and their difference can be computed on-the-fly,
cell by cell, rather than in a postprocessing step. The same reasoning holds
for the pairs of coarse and fine models within multilevel MCMC.

## Localisation

ExaHyPE couples volumetrically. That does not mean that all solvers have to be
solved everywhere. Indeed, the idea to offer support for multiple solvers arose
originally from the a posteriori limiting of Dumbser et al where a Finite Volume
solver complements ADER-DG solutions where shocks arise.

Most projects that follow these rationale realise the same pattern:
We define two different solvers, and we couple them
with each other. The coupling (and solve) however are local operations:

-   The low-order (typically Finite Volume - we therefore always refer
    to Finite Volumes or FV from hereon) solver is our fallback strategy and we therefore
    neither solve with Finite Volumes everywhere in the computational
    domain nor should do we store the corresponding data.

-   The higher-order scheme is used everywhere where the FV is not
    applied or labelled as valid.

-   The FV and the higher-order solver are not coupled everywhere but only in
    boundary regions.

We will effectively end up with a domain decomposition where parts of
the computational domain are handled by the higher-order solver and the other
parts are handled with FV.
To realise this, we need a marker or oracle, i.e. a magic function
per cell which tells us, similar to a multi-phase flow, which
solver rules in this particular cell.
The
marker will guide us which solver is "valid" in which cell, where one solver
overrules the other one, and where we do not have to store one solver's data
or the other.

The image below illustrates the core idea behind typically coupling approaches
in ExaHyPE. The sizes of the overlap might differ, but the principle is always
the same. The illustration shows the minimal overlap that we can typically
use. Both the upper and the lower mesh illustrate the same mesh. I use colours
to show which solver exists in which mesh cell.

![Overlap of two localised solvers](illustrations/44_localisation.png)

- Solver A is running within the filled cells visualised on the the top. In
  empty cells, A neither solves anything nor does it hold data.
- Solver B is running within the filled cells visualised on the bottom. In
  empty cells, B neither solves anything nor does it hold data.
- Consequently, dark red cells are exclusively solved by A, whereas dark
  green cells are exclusively B.
- In orange cells, A solves its PDE. Afterwards, A's updated solution is
  projected onto B's solution. That is, the blue cells hold B's data, but
  they are not updated by B but instead hold projections of A's data. They
  kind of serve as boundary layer for B.
- In the yellow cells, A does not solve its PDE but holds A's data. After
  each step, the cell's solution by B (light green) is written into A's data.
  The yellow cells serve as Dirichlet boundary data for A's solve.

Starting from this description, sophisticated coupling equations can be
realised, where you exploit the fact that we have a volumetric overlap.

### Localisation guided by a marker

Skip this section if you prefer a hard-coded marking of active regions.

### Avoid that solver data is held everywhere

Once you know where a solver is valid, it makes sense also to only store
the solver's data in the valid region. Otherwise, a lot of memory is "wasted"
which furthermore will be moved/accessed in each and every grid sweep, as the
mesh management of Peano so far does not know that ExaHyPE might not need the
data. ExaHyPE has to let Peano know.

The technical principles behind such a localisation @ref page_peano_localisation "are discussed on a generic Peano page of its own".
ExaHyPE builds up pretty deep inheritance hierarchy
where every subclass specialises and augments the basic numerical scheme, and
also implements additional guards when data is stored or not.
However, most of these subclasses rely on only six routines to control
the data flow:

- _provide_cell_data_to_compute_kernels_default_guard(),
- _load_cell_data_default_guard(),
- _store_cell_data_default_guard(),
- _provide_face_data_to_compute_kernels_default_guard(),
- _load_face_data_default_guard(),
- _store_face_data_default_guard(),

The implementation then uses these predicates to puzzle the actual storage
scheme together via calls similar to

```python
        self._patch.generator.load_store_compute_flag = "::peano4::grid::constructLoadStoreComputeFlag({},{},{})".format(
            self._provide_cell_data_to_compute_kernels_default_guard(),
            self._load_cell_data_default_guard(),
            self._store_cell_data_default_guard(),
        )
```

By default, most solvers only store data for unrefined
cells. They mask coarser levels with in the tree out.
Subclasses such as enclave tasking variations augment the guards
further such that not all data are stored all the time. Nothing stops you
from adding further constraints and to use a solver only in a certain
subdomain for example.

## References

- M. B. Giles: [Multilevel Monte Carlo path simulation](https://doi.org/10.1287/opre.1070.0496).
  Operations Research, 56(3), pp. 607-617 (2008)
- M. B. Giles: [Multilevel Monte Carlo methods](https://doi.org/10.1017/S096249291500001X).
  Acta Numerica, 24, pp. 259-328 (2015)
- S. Mishra, Ch. Schwab, J. Šukys: [Multi-level Monte Carlo finite volume methods for nonlinear systems of conservation laws in multi-dimensions](https://doi.org/10.1016/j.jcp.2012.01.011).
  Journal of Computational Physics, 231(8), pp. 3365-3388 (2012)
- A.-L. Haji-Ali, F. Nobile, R. Tempone: [Multi-index Monte Carlo: when sparsity meets sampling](https://doi.org/10.1007/s00211-015-0734-5).
  Numerische Mathematik, 132, pp. 767-806 (2016)
- T. J. Dodwell, C. Ketelsen, R. Scheichl, A. L. Teckentrup: [A hierarchical multilevel Markov chain Monte Carlo algorithm with applications to uncertainty quantification in subsurface flow](https://doi.org/10.1137/130915005).
  SIAM/ASA Journal on Uncertainty Quantification, 3(1), pp. 1075-1108 (2015)
- M. B. Lykkegaard, T. J. Dodwell, C. Fox, G. Mingas, R. Scheichl: [Multilevel delayed acceptance MCMC](https://doi.org/10.1137/22M1476770).
  SIAM/ASA Journal on Uncertainty Quantification, 11, pp. 1-30 (2023)
- L. Seelinger, A. Reinarz, L. Rannabauer, M. Bader, P. Bastian, R. Scheichl: [High performance uncertainty quantification with parallelized multilevel Markov chain Monte Carlo](https://doi.org/10.1145/3458817.3476150).
  Proceedings of the International Conference for High Performance Computing, Networking, Storage and Analysis (SC21), pp. 1-15 (2021)
- A. Reinarz, D. E. Charrier, M. Bader et al.: [ExaHyPE: An engine for parallel dynamically adaptive simulations of wave problems](https://doi.org/10.1016/j.cpc.2020.107251).
  Computer Physics Communications, 254, 107251 (2020)
- B. Peherstorfer, K. Willcox, M. Gunzburger: [Survey of multifidelity methods in uncertainty propagation, inference, and optimization](https://doi.org/10.1137/16M1082469).
  SIAM Review, 60(3), pp. 550-591 (2018)